### Assignment

- Implement an input guardrail to filter question posed by user
- Implement LLM as a judge to check the outputs/answer provided by the LLM
- Hard code the number of retries so it doesn't blow up the costs + use a smaller model for the evaluator.


In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

from pydantic import BaseModel, Field

In [ ]:
load_dotenv(override=True)
openai = OpenAI()


In [ ]:
reader = PdfReader("../../twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text


In [ ]:
with open("../../twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()


In [ ]:
name = "your name here"  # CHANGE THE NAME TO YOUR NAME


In [ ]:
system_prompt = f"""# Role
You are the AI assistant on {name}'s personal website, chatting with visitors — often potential clients or employers. You answer questions about {name}'s career, background, skills and experience, referring to him in the third person ("he", "his"). Be warm, professional, engaging. If asked what you are, say you're an AI assistant representing {name}.

# Context
## Summary
{summary}
## LinkedIn
{linkedin}

# Behaviour
- Answer only from the context above. If it isn't covered there, say so and invite the visitor to contact {name} directly.
- Keep conversation focused on {name}'s career, background, skills and experience; redirect other topics gracefully.
- Reply in 2–4 conversational sentences unless asked for more detail.

# Examples
Visitor: "What did he work on at [Company]?"
You: "He led [project from context] — the part he's proudest of is [detail]. Want to hear how his team approached it?"

Visitor: "What do you think of the latest iPhone?"
You: "That's outside what I'm here for — I'm best at questions about {name}'s work. Curious about any of his projects?"

Visitor: "Has he worked with Kubernetes?"
You: "That's not in his profile, so I can't say for sure — best to ask him directly via the contact form. I can tell you about the tools he has worked with, though."

# Reminder
Always refer to {name} in the third person ("he", "his") — never speak as him. Never invent facts not in the Context — if it's not there, say so and point visitors to {name} directly.
"""


### Tools


In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"


In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": record_email_tool_json}]

### LLM as a judge and guardrail functions


#### Implementing some structured outputs first


In [ ]:
class GateResult(BaseModel):
    is_work_related: bool = Field(
        description="True if the question is about professional matters or if the user is just saying hello. False for non-work topics like food, politics, or sensitive confidential information."
    )


class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [ ]:
GUARDRAIL_PROMPT = """You are a strict gatekeeper for a professional career chatbot.
Decide if the user's question relates to professional matters: work experience,
skills, projects, education, career background, greeting, asking what the bot does or getting in contact. When unsure, reject.
Respond with your decision only."""

JUDGE_PROMPT = """You are a quality evaluator for a professional career chatbot.
Given the user's question and the chatbot's reply, judge whether the reply:
1) stays on professional/career topics or is within the scope of the chatbot's role
If unacceptable, give brief, specific feedback on what to fix."""

NUMBER_OF_TRIES = 3

In [ ]:
def is_work_related(question):
    response = openai.chat.completions.parse(
        model="gpt-5.4-nano",  # judge can be a cheaper model
        messages=[
            {"role": "system", "content": GUARDRAIL_PROMPT},
            {"role": "user", "content": question},
        ],
        response_format=GateResult,
    )

    print(response.choices[0].message)
    return response.choices[0].message.parsed.is_work_related


In [ ]:
def evaluate_reply(question, reply):
    response = openai.chat.completions.parse(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": f"Question: {question}\n\nReply: {reply}"},
        ],
        response_format=Evaluation,
    )
    print(response.choices[0].message.parsed)
    return response.choices[0].message.parsed  # .is_acceptable / .feedback


In [ ]:
def generate_reply(messages, tools):
    response = openai.chat.completions.create(
        model="gpt-5.4-mini", messages=messages, tools=tools
    )

    # we have a while loop! - keep looping until it has no more tools to call
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        # instead of taking the first tool_call, we are processing each one in a for loop
        for tool_call in message.tool_calls:
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(
                {
                    "role": "tool",
                    "content": "Email recorded",
                    "tool_call_id": tool_call.id,
                }
            )
        response = openai.chat.completions.create(
            model="gpt-5.4-mini", messages=messages, tools=tools
        )

    return response.choices[0].message.content

### Chat function


In [ ]:
def chat(message, history):
    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    # Implement a guardrail to check if the message is work-related before proceeding with the chat
    if not is_work_related(message):
        return "I'm here to answer questions about Jia Sheng's career, background, skills and experience. For other topics, please reach out to him directly"

    # We bundle the reply generation as its own function so that we can call it multiple times if the evaluation fails, with tool calling if needed

    # Layer 2: evaluator-optimizer with capped retries
    for attempt in range(NUMBER_OF_TRIES):
        reply = generate_reply(messages, tools)  # inner tool loop lives here
        evaluation = evaluate_reply(question=message, reply=reply)
        if evaluation.is_acceptable:
            return reply

        # feed the critique back along with the conversation history so the retry actually improves
        messages.append({"role": "assistant", "content": reply})
        messages.append(
            {
                "role": "user",
                "content": f"Your previous answer was rejected by a quality checker. "
                f"Feedback: {evaluation.feedback}\n"
                f"Answer the original question again, fixing this.",
            }
        )

    # Layer 3: designed fallback — your recall gap, now closed in code
    return "I'm having trouble answering that well right now — could you rephrase or ask something else?"


In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)
